In [1]:
# EZ-1 — Environment (fresh runtime). Afterwards: Runtime → Restart session, then run EZ-2 onward.
!apt-get -qq install -y git-lfs > /dev/null
import os, subprocess
from pathlib import Path

ROOT = Path("/content/Project-phase-2")
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/CMU-18734-17731-2026-Fall/Project-phase-2.git", str(ROOT)], check=True)
subprocess.run(["git", "lfs", "install"], check=True)
subprocess.run(["git", "lfs", "pull"], cwd=ROOT, check=True)

EZ_DIR = Path("/content/ez-mia")
EZ_COMMIT = "47fc36dc8540fa05f4b89d0dac2dc0130e6856af"
if not EZ_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/JetBrains-Research/ez-mia.git", str(EZ_DIR)], check=True)
subprocess.run(["git", "checkout", "-q", EZ_COMMIT], cwd=EZ_DIR, check=True)

%pip install -q "transformers==4.56.2" "peft==0.13.2" "accelerate==1.10.1" "datasets==4.1.1"
print("Done. Restart the session now.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requir

In [1]:
# EZ-2 — Imports and explicit constants. Nothing is inherited from earlier runtimes.
import os, json, io, hashlib, importlib.util, random
os.environ["USE_TF"] = "0"; os.environ["USE_FLAX"] = "0"
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import transformers, peft
from peft import PeftModel
from sklearn.metrics import roc_auc_score, roc_curve
from transformers import AutoModelForCausalLM, AutoTokenizer

assert transformers.__version__ == "4.56.2" and peft.__version__ == "0.13.2", (transformers.__version__, peft.__version__)

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/Project-phase-2")
DRIVE = Path("/content/drive/MyDrive/MIA_phase2")
OLD_CACHE = DRIVE / "cache"                       # read-only for this notebook
EZ_CACHE = DRIVE / "cache" / "ezmia"              # all new EZ-MIA files go here
EZ_CACHE.mkdir(parents=True, exist_ok=True)
EZ_DIR = Path("/content/ez-mia")
EZ_COMMIT = "47fc36dc8540fa05f4b89d0dac2dc0130e6856af"

MODEL_NAME = "gpt2_3_lora32_adamw_b8_lr2"
MAX_LENGTH = 512                                  # same as cached losses and target training
BATCH_SIZE = 2                                    # same batching as the cached losses
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Use a GPU runtime: cached losses were computed in fp16 on CUDA."
model_dtype = torch.float16                       # matches the cached computation

assert ROOT.exists() and DRIVE.exists() and EZ_DIR.exists()
head = os.popen(f"git -C {EZ_DIR} rev-parse HEAD").read().strip()
assert head == EZ_COMMIT, head
print("Device:", torch.cuda.get_device_name(0), "| EZ-MIA commit:", head[:7])

Mounted at /content/drive
Device: NVIDIA L4 | EZ-MIA commit: 47fc36d


In [2]:
# EZ-3 — Dev data and tokenizer, exactly as in the attack notebook.
def read_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)

DEV_TEXTS = read_json(ROOT / "data" / "train" / "test.json")
LABELS = np.asarray(read_json(ROOT / "data" / "train" / "test_label.json"), dtype=np.int64)
assert len(DEV_TEXTS) == len(LABELS) == 2000 and all(isinstance(t, str) for t in DEV_TEXTS)
assert set(np.unique(LABELS)) == {0, 1} and LABELS.sum() == 1000

ADAPTER_DIR = ROOT / "models" / "train" / MODEL_NAME
adapter_config = read_json(ADAPTER_DIR / "adapter_config.json")
BASE_MODEL_NAME = adapter_config["base_model_name_or_path"]

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

DEV_ORDER_SHA = hashlib.sha256("\x1e".join(DEV_TEXTS).encode("utf-8")).hexdigest()
print("Base model:", BASE_MODEL_NAME, "| dev order fingerprint:", DEV_ORDER_SHA[:16])

Base model: gpt2 | dev order fingerprint: e415836c89b99165


In [3]:
# EZ-4 — Reuse cached per-token true-token losses (target and pretrained GPT-2, both fp16 on CUDA).
def split_concat(z, key):
    return [a.astype(np.float32) for a in np.split(z[key], np.cumsum(z["lengths"])[:-1])]

_z = np.load(OLD_CACHE / "train_base_target_losses.npz")
LENGTHS = _z["lengths"].astype(np.int64)
TGT_LOSS_CACHED = split_concat(_z, "target_losses")
BASE_LOSS_CACHED = split_concat(_z, "base_losses")      # LoRA adapter disabled = pretrained GPT-2
assert len(LENGTHS) == len(TGT_LOSS_CACHED) == len(BASE_LOSS_CACHED) == 2000

# Scored positions per text must equal (truncated token count − 1) under our tokenization.
enc = tokenizer(DEV_TEXTS, truncation=True, max_length=MAX_LENGTH)["input_ids"]
expected = np.array([len(ids) - 1 for ids in enc], dtype=np.int64)
assert np.array_equal(expected, LENGTHS), "cache does not align with current tokenization/order"
assert all(a.shape == b.shape == (n,) for a, b, n in zip(TGT_LOSS_CACHED, BASE_LOSS_CACHED, LENGTHS))
print(f"Cached losses aligned: 2000 texts, {LENGTHS.sum()} scored tokens, min/max length {LENGTHS.min()}/{LENGTHS.max()}")

Cached losses aligned: 2000 texts, 299281 scored tokens, min/max length 24/511


In [4]:
# EZ-5 — Load frozen LR4 dev scores and verify alignment with DEV_TEXTS / LABELS.
def tpr_strict(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    return float(np.max(tpr[fpr < 0.01]))

S4 = np.load(OLD_CACHE / "staged4_dev_scores_4pairs.npz")
print("Keys in frozen stage file:", list(S4.keys()))
LR4_KEY = "LR (all OUT, κ=8 fixed)"
LR4_DEV = S4[LR4_KEY].astype(np.float64)
B5_SAVED = S4["B5"].astype(np.float64)
assert LR4_DEV.shape == B5_SAVED.shape == (2000,) and np.isfinite(LR4_DEV).all()

settings = read_json(DRIVE / "frozen_LR4_k8" / "settings.json")
assert np.isclose(tpr_strict(LABELS, LR4_DEV), settings["dev"]["LR"], atol=1e-9)
assert np.isclose(tpr_strict(LABELS, B5_SAVED), settings["dev"]["B5"], atol=1e-9)

# Independent order check: rebuild B5 from per-text caches in DEV_TEXTS order.
def load_ref(fname, key="losses"):
    return split_concat(np.load(OLD_CACHE / fname), key)
fixed_refs = [load_ref("train_shadow_1001_losses.npz", "shadow_losses"),
              load_ref("train_shadow_1002_losses.npz"),
              load_ref("train_ref_val_losses.npz"),
              load_ref("train_ref_final_losses.npz")]
pair_refs = {n: load_ref(f"train_shadow_{n}_losses.npz") for n in ("inA", "inB")}
norm = lambda t: " ".join(t.split())
pair_sets = {n: {norm(r["text"]) for r in read_json(DRIVE / "data" / f"shadow_{n}" / "train_finetune.json")}
             for n in ("inA", "inB")}
for arrs in fixed_refs + list(pair_refs.values()):
    assert len(arrs) == 2000 and all(a.shape == t.shape for a, t in zip(arrs, TGT_LOSS_CACHED))

b5 = np.empty(2000)
for i, t in enumerate(TGT_LOSS_CACHED):
    outs = [r[i] for r in fixed_refs] + [pair_refs[n][i] for n in ("inA", "inB")
                                          if norm(DEV_TEXTS[i]) not in pair_sets[n]]
    d = np.stack(outs).mean(0) - t
    b5[i] = d.mean() * np.sqrt(t.size)
assert np.allclose(b5, B5_SAVED, rtol=0, atol=1e-5), f"max diff {np.abs(b5 - B5_SAVED).max()}"
print(f"Alignment verified. LR4 dev: AUC={roc_auc_score(LABELS, LR4_DEV):.4f} | "
      f"strict TPR@FPR<1%={tpr_strict(LABELS, LR4_DEV):.4f}")

Keys in frozen stage file: ['B5', 'B_ext', 'midpoint (all OUT)', 'LR (all OUT, κ=8 fixed)', 'LR (all OUT, κ*=16 mock)']
Alignment verified. LR4 dev: AUC=0.9744 | strict TPR@FPR<1%=0.7120


In [5]:
# EZ-6 — Fresh dev-target pass: true-token log-prob, top-1 id, target id per scored position. Cached once.
EZ_STATS = EZ_CACHE / "ezmia_dev_target_stats.npz"

if EZ_STATS.exists():
    zs = np.load(EZ_STATS)
    assert zs["dev_order_sha"].item() == DEV_ORDER_SHA
    cuts = np.cumsum(zs["lengths"])[:-1]
    TGT_LP = np.split(zs["tgt_lp"], cuts); TOP1 = np.split(zs["top1"], cuts); TARGET_IDS = np.split(zs["target_ids"], cuts)
    print("Loaded cached EZ-MIA target stats.")
else:
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=model_dtype)
    target = PeftModel.from_pretrained(base, ADAPTER_DIR)
    target.config.pad_token_id = tokenizer.pad_token_id
    target.to(device).eval()

    TGT_LP, TOP1, TARGET_IDS = [], [], []
    with torch.inference_mode():
        for s in tqdm(range(0, 2000, BATCH_SIZE)):
            batch = tokenizer(DEV_TEXTS[s:s + BATCH_SIZE], return_tensors="pt", padding=True,
                              truncation=True, max_length=MAX_LENGTH)
            batch = {k: v.to(device) for k, v in batch.items()}
            tids = batch["input_ids"][:, 1:]
            mask = batch["attention_mask"][:, 1:].bool()
            logits = target(**batch, use_cache=False).logits[:, :-1, :]
            lp = F.log_softmax(logits.float(), dim=-1)                       # as in official _extract_stats_for_model
            correct = lp.gather(-1, tids.unsqueeze(-1)).squeeze(-1)
            top1 = torch.topk(lp, k=1, dim=-1).indices.squeeze(-1)
            for b in range(tids.shape[0]):
                m = mask[b]
                TGT_LP.append(correct[b][m].cpu().numpy().astype(np.float32))
                TOP1.append(top1[b][m].cpu().numpy().astype(np.int32))
                TARGET_IDS.append(tids[b][m].cpu().numpy().astype(np.int32))
            del logits, lp
    target.to("cpu"); del target, base; torch.cuda.empty_cache()

    lens = np.array([len(a) for a in TGT_LP], dtype=np.int64)
    np.savez_compressed(EZ_STATS, lengths=lens, tgt_lp=np.concatenate(TGT_LP), top1=np.concatenate(TOP1),
                        target_ids=np.concatenate(TARGET_IDS), dev_order_sha=np.array(DEV_ORDER_SHA),
                        note=np.array("dev target (LoRA GPT-2), fp16 forward, log_softmax in fp32; "
                                      "positions = attention_mask[:,1:]; batch 2, right padding, max_length 512"))
    print("Saved", EZ_STATS)

# Alignment with the cached losses and the tokenizer
assert len(TGT_LP) == 2000 and np.array_equal([len(a) for a in TGT_LP], LENGTHS)
for i in range(2000):
    assert np.array_equal(TARGET_IDS[i], np.asarray(enc[i][1:], dtype=np.int32))
max_diff = max(np.abs(-TGT_LP[i] - TGT_LOSS_CACHED[i]).max() for i in range(2000))
print(f"Max |fresh target loss − cached target loss| = {max_diff:.2e}")
assert max_diff < 1e-3, "fresh pass does not reproduce cached target losses"

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Saved /content/drive/MyDrive/MIA_phase2/cache/ezmia/ezmia_dev_target_stats.npz
Max |fresh target loss − cached target loss| = 0.00e+00


In [7]:
# EZ-7 — EZ-MIA score via the official function (loaded directly from the pinned file), plus a numpy port.
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod
ez_official = load_module("ez_score_official", EZ_DIR / "mia" / "ez_score.py").error_zone_pos_neg_sum_ratio
tpr_at_fpr_official = load_module("metrics_official", EZ_DIR / "mia" / "metrics.py").tpr_at_fpr

def ez_numpy(t_lp, r_lp, top1, tids, ignore_bos=True, min_tokens=2):
    valid = np.ones(t_lp.size, dtype=bool)
    if ignore_bos and valid.size:
        valid[0] = False
    err = (top1 != tids) & valid
    if err.sum() < min_tokens:
        return 0.0
    d = (t_lp.astype(np.float32) - r_lp.astype(np.float32))[err]
    pos, neg = d[d > 0].sum(dtype=np.float32), np.abs(d[d < 0]).sum(dtype=np.float32)
    if neg == 0:
        return 0.0
    v = float(pos / (neg + np.float32(1e-12)))
    return 0.0 if (v != v or v < 0) else v

ez_off, ez_np, n_err = [], [], []
for i in range(2000):
    t_lp = TGT_LP[i]                      # target log p(true token), fresh pass (same pass as top-1)
    r_lp = -BASE_LOSS_CACHED[i]           # pretrained GPT-2 log p(true token), cached
    mask = torch.ones(t_lp.size)          # arrays hold valid positions only
    ez_off.append(ez_official(torch.from_numpy(t_lp), torch.from_numpy(r_lp),
                              torch.from_numpy(TOP1[i]).long(), torch.from_numpy(TARGET_IDS[i]).long(),
                              mask, ignore_bos=True, min_tokens=2))
    ez_np.append(ez_numpy(t_lp, r_lp, TOP1[i], TARGET_IDS[i]))
    n_err.append(int(((TOP1[i] != TARGET_IDS[i])[1:]).sum()))

EZ_DEV = np.nan_to_num(np.array(ez_off, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)  # as in attack.py
assert np.allclose(EZ_DEV, np.array(ez_np, dtype=np.float32), rtol=1e-5, atol=1e-6), "port disagrees with official"
n_err = np.array(n_err)
print(f"Scores: min={EZ_DEV.min():.4f} max={EZ_DEV.max():.4f} | zeros: members={int((EZ_DEV[LABELS==1]==0).sum())}, "
      f"non-members={int((EZ_DEV[LABELS==0]==0).sum())} | unique values={np.unique(EZ_DEV).size}")
print(f"Error-zone tokens per text: median {np.median(n_err):.0f} "
      f"(fraction of scored tokens: {np.median(n_err / np.maximum(LENGTHS - 1, 1)):.2f})")

Scores: min=0.5071 max=33.2714 | zeros: members=0, non-members=0 | unique values=2000
Error-zone tokens per text: median 81 (fraction of scored tokens: 0.61)


In [8]:
# EZ-8 — Dev AUC and strict TPR@FPR<0.01: EZ-MIA (pretrained GPT-2 reference) vs frozen LR4. Save results.
assert EZ_DEV.shape == (2000,) and np.isfinite(EZ_DEV).all()

def strict_tpr_fast(s, y):
    neg, pos = s[y == 0], s[y == 1]
    v = np.sort(neg)[::-1][int(np.ceil(0.01 * neg.size)) - 1]
    return np.mean(pos > v)
for s in (EZ_DEV.astype(np.float64), LR4_DEV):
    assert np.isclose(strict_tpr_fast(s, LABELS), tpr_strict(LABELS, s))

rng = np.random.default_rng(20261017)
pos_i, neg_i = np.flatnonzero(LABELS == 1), np.flatnonzero(LABELS == 0)
boot = [np.concatenate([rng.choice(pos_i, pos_i.size), rng.choice(neg_i, neg_i.size)]) for _ in range(2000)]
d = np.array([strict_tpr_fast(EZ_DEV[ix].astype(np.float64), LABELS[ix]) - strict_tpr_fast(LR4_DEV[ix], LABELS[ix])
              for ix in boot])

rows = []
for name, s in (("EZ-MIA (base GPT-2 ref, official score)", EZ_DEV.astype(np.float64)),
                ("LR4 (frozen, κ=8, 4 pairs)", LR4_DEV)):
    rows.append({"method": name, "dev AUC": roc_auc_score(LABELS, s),
                 "strict TPR@FPR<0.01": tpr_strict(LABELS, s),
                 "repo interp TPR@1%FPR": tpr_at_fpr_official(LABELS, s, 0.01)})
res = pd.DataFrame(rows)
lo, hi = np.percentile(d, [2.5, 97.5])
print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nEZ-MIA − LR4, strict TPR: {rows[0]['strict TPR@FPR<0.01'] - rows[1]['strict TPR@FPR<0.01']:+.4f} "
      f"| paired stratified bootstrap 95% CI [{lo:+.3f}, {hi:+.3f}] | bootstrap fraction Δ>0: {np.mean(d > 0):.3f}")

out_scores = EZ_CACHE / "ezmia_dev_scores.npz"
out_csv = EZ_CACHE / "ezmia_dev_vs_LR4.csv"
out_notes = EZ_CACHE / "ezmia_notes.json"
for p in (out_scores, out_csv, out_notes):
    assert "ezmia" in p.name and "LR4_k8" not in p.name
np.savez(out_scores, ezmia_dev=EZ_DEV, lr4_dev=LR4_DEV, labels=LABELS, dev_order_sha=np.array(DEV_ORDER_SHA))
res.to_csv(out_csv, index=False)
json.dump({
    "official_repo": "https://github.com/JetBrains-Research/ez-mia", "commit": EZ_COMMIT,
    "score": "error_zone_pos_neg_sum_ratio(ignore_bos=True, min_tokens=2): over positions where target top-1 != "
             "true token (first scored position dropped), sum(Δ>0)/sum(|Δ<0|), Δ = log p_target − log p_ref; "
             "0 if <2 error tokens, no negative Δ, or NaN/negative; float32 + nan_to_num as in attack.py; "
             "higher = member",
    "reference": "base variant: pretrained GPT-2 (cached, PEFT adapter disabled)",
    "inputs_reused": "train_base_target_losses.npz base_losses (ref log-probs); target log-probs re-derived in "
                     "the fresh pass and checked against cached target_losses",
    "new_inference": "one dev-target forward pass for top-1 ids (ezmia_dev_target_stats.npz)",
    "departures": [
        "sequence length 512 (target training block size; cached losses) instead of 128 in the repo configs",
        "target is the course LoRA GPT-2 (r=32, 3 epochs), not trained by the repo pipeline",
        "fp16 forward pass (to match cached losses) instead of fp32",
        "pad token = EOS instead of an added <|pad|> token; padding is masked in both",
        "primary metric is strict TPR at FPR<0.01 (competition); repo's interpolated tpr_at_fpr also reported",
    ],
    "no_validation_use": True,
}, open(out_notes, "w"), indent=2)
print("Saved:", out_scores.name, out_csv.name, out_notes.name, "→", EZ_CACHE)

                                 method  dev AUC  strict TPR@FPR<0.01  repo interp TPR@1%FPR
EZ-MIA (base GPT-2 ref, official score)   0.6829               0.0120                 0.0150
             LR4 (frozen, κ=8, 4 pairs)   0.9744               0.7120                 0.7290

EZ-MIA − LR4, strict TPR: -0.7000 | paired stratified bootstrap 95% CI [-0.773, -0.581] | bootstrap fraction Δ>0: 0.000
Saved: ezmia_dev_scores.npz ezmia_dev_vs_LR4.csv ezmia_notes.json → /content/drive/MyDrive/MIA_phase2/cache/ezmia


In [9]:
# EZ-9 — EZ-MIA with an in-domain non-member reference (repo's "sft" variant, using cached shadows).
# Same official scoring function; compared with plain B using the identical single reference. Dev only.
sft_refs = {"shadow1001": load_ref("train_shadow_1001_losses.npz", "shadow_losses"),
            "shadow1002": load_ref("train_shadow_1002_losses.npz")}

rows, saved = [], {}
for name, R in sft_refs.items():
    assert len(R) == 2000 and all(r.shape == t.shape for r, t in zip(R, TGT_LP))
    ez = np.array([ez_official(torch.from_numpy(TGT_LP[i]), torch.from_numpy(-R[i]),
                               torch.from_numpy(TOP1[i]).long(), torch.from_numpy(TARGET_IDS[i]).long(),
                               torch.ones(TGT_LP[i].size), ignore_bos=True, min_tokens=2)
                   for i in range(2000)], dtype=np.float32)
    ez = np.nan_to_num(ez, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
    b1 = np.array([(R[i] - TGT_LOSS_CACHED[i]).mean() * np.sqrt(LENGTHS[i]) for i in range(2000)])
    saved[f"ezmia_sft_{name}"] = ez; saved[f"B_1ref_{name}"] = b1
    for label, s in ((f"EZ-MIA, sft ref = {name}", ez), (f"B (mean gap × √T), ref = {name}", b1)):
        d = np.array([strict_tpr_fast(s[ix], LABELS[ix]) - strict_tpr_fast(LR4_DEV[ix], LABELS[ix]) for ix in boot])
        lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"method": label, "dev AUC": roc_auc_score(LABELS, s),
                     "strict TPR@FPR<0.01": tpr_strict(LABELS, s),
                     "Δ vs LR4": tpr_strict(LABELS, s) - tpr_strict(LABELS, LR4_DEV),
                     "95% CI vs LR4": f"[{lo:+.3f}, {hi:+.3f}]"})
rows.append({"method": "LR4 (frozen, κ=8, 4 pairs)", "dev AUC": roc_auc_score(LABELS, LR4_DEV),
             "strict TPR@FPR<0.01": tpr_strict(LABELS, LR4_DEV), "Δ vs LR4": 0.0, "95% CI vs LR4": ""})
t9 = pd.DataFrame(rows)
print(t9.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

out = EZ_CACHE / "ezmia_sft_dev_scores.npz"
assert "LR4_k8" not in out.name
np.savez(out, labels=LABELS, dev_order_sha=np.array(DEV_ORDER_SHA), **saved)
t9.to_csv(EZ_CACHE / "ezmia_sft_dev_vs_LR4.csv", index=False)
print("Saved", out.name)

                             method  dev AUC  strict TPR@FPR<0.01  Δ vs LR4    95% CI vs LR4
       EZ-MIA, sft ref = shadow1001   0.9234               0.2380   -0.4740 [-0.618, -0.367]
B (mean gap × √T), ref = shadow1001   0.9390               0.4420   -0.2700 [-0.438, -0.144]
       EZ-MIA, sft ref = shadow1002   0.9261               0.2150   -0.4970 [-0.646, -0.297]
B (mean gap × √T), ref = shadow1002   0.9394               0.4490   -0.2630 [-0.338, -0.122]
         LR4 (frozen, κ=8, 4 pairs)   0.9744               0.7120    0.0000                 
Saved ezmia_sft_dev_scores.npz


In [10]:
# EZ-V — Empirical verification of EZ-MIA details against the official implementation. Dev only, no validation.
from peft import PeftModel as _Peft
rng_v = np.random.default_rng(7)
batch_starts = np.sort(rng_v.choice(np.arange(0, 2000, BATCH_SIZE), size=32, replace=False))  # reproduce original batches
SAMPLE = np.concatenate([np.arange(s, s + BATCH_SIZE) for s in batch_starts])

def forward_stats(model, idx_starts):
    lp_out, top1_out, tid_out = {}, {}, {}
    with torch.inference_mode():
        for s in idx_starts:
            batch = tokenizer(DEV_TEXTS[s:s + BATCH_SIZE], return_tensors="pt", padding=True,
                              truncation=True, max_length=MAX_LENGTH)
            batch = {k: v.to(device) for k, v in batch.items()}
            tids = batch["input_ids"][:, 1:]; mask = batch["attention_mask"][:, 1:].bool()
            lp = F.log_softmax(model(**batch, use_cache=False).logits[:, :-1, :].float(), dim=-1)
            corr = lp.gather(-1, tids.unsqueeze(-1)).squeeze(-1)
            top1 = torch.topk(lp, k=1, dim=-1).indices.squeeze(-1)
            for b in range(tids.shape[0]):
                m = mask[b]
                lp_out[s + b] = corr[b][m].cpu().numpy(); top1_out[s + b] = top1[b][m].cpu().numpy()
                tid_out[s + b] = tids[b][m].cpu().numpy()
    return lp_out, top1_out, tid_out

# ---- 1. Target has its LoRA adapter enabled ----
assert BASE_MODEL_NAME == "gpt2"
tgt = _Peft.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=model_dtype), ADAPTER_DIR)
tgt.config.pad_token_id = tokenizer.pad_token_id; tgt.to(device).eval()
lora_B_norm = sum(p.float().norm().item() for n, p in tgt.named_parameters() if "lora_B" in n)
print("Target: PeftModel =", isinstance(tgt, _Peft), "| active adapter:", tgt.active_adapter,
      "| total ||lora_B|| =", round(lora_B_norm, 3))
assert isinstance(tgt, _Peft) and lora_B_norm > 0
t_lp, t_top1, t_tid = forward_stats(tgt, batch_starts)

# Causal shift check: log p(token k+1 | tokens ≤ k) from a prefix-only run equals position k of the full run.
with torch.inference_mode():
    for i in SAMPLE[:3]:
        ids = torch.tensor(enc[i], device=device).unsqueeze(0)
        for k in (5, min(50, ids.shape[1] - 2)):
            lp_prefix = F.log_softmax(tgt(input_ids=ids[:, :k + 1]).logits[0, -1].float(), -1)[ids[0, k + 1]].item()
            assert abs(lp_prefix - t_lp[i][k]) < 2e-2, (i, k, lp_prefix, t_lp[i][k])
print("Causal shift verified: position k scores token k+1 given tokens 0..k.")
tgt.to("cpu"); del tgt; torch.cuda.empty_cache()

# ---- 2. Reference is plain pretrained GPT-2 (no PEFT wrapper) ----
ref = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=model_dtype)
assert not isinstance(ref, _Peft) and not any("lora" in n for n, _ in ref.named_parameters())
ref.config.pad_token_id = tokenizer.pad_token_id; ref.to(device).eval()
r_lp, _, _ = forward_stats(ref, batch_starts)
ref.to("cpu"); del ref; torch.cuda.empty_cache()

d_tgt = max(np.abs(t_lp[i] - TGT_LP[i]).max() for i in SAMPLE)
d_ref = max(np.abs(r_lp[i] + BASE_LOSS_CACHED[i]).max() for i in SAMPLE)
top1_agree = np.mean(np.concatenate([t_top1[i] == TOP1[i] for i in SAMPLE]))
gap = np.mean(np.concatenate([np.abs(t_lp[i] - r_lp[i]) for i in SAMPLE]))
print(f"Target re-run vs EZ-6 stats: max |Δlp| = {d_tgt:.2e}, top-1 agreement = {top1_agree:.4f}")
print(f"Plain GPT-2 vs cached reference (adapter-disabled) losses: max |Δ| = {d_ref:.2e}")
print(f"Mean |target − reference| log-prob = {gap:.3f} (adapter has an effect)")
assert d_tgt < 1e-3 and top1_agree > 0.999 and d_ref < 1e-3 and gap > 1e-2
assert all(np.array_equal(t_tid[i], np.asarray(enc[i][1:])) for i in SAMPLE)

# ---- 3. Padding exclusion: official function gives identical scores with padded, masked inputs ----
for i in SAMPLE[:16]:
    n, pad = TGT_LP[i].size, 37
    tp = torch.from_numpy(np.concatenate([TGT_LP[i], np.full(pad, 5.0, np.float32)]))
    rp = torch.from_numpy(np.concatenate([-BASE_LOSS_CACHED[i], np.full(pad, -9.0, np.float32)]))
    t1 = torch.from_numpy(np.concatenate([TOP1[i], np.zeros(pad, np.int32)])).long()
    ti = torch.from_numpy(np.concatenate([TARGET_IDS[i], np.ones(pad, np.int32)])).long()   # pads look like errors
    mk = torch.cat([torch.ones(n), torch.zeros(pad)])
    padded = ez_official(tp, rp, t1, ti, mk, ignore_bos=True, min_tokens=2)
    assert np.isclose(np.float32(padded), EZ_DEV[i], rtol=1e-6, atol=1e-7), (i, padded, EZ_DEV[i])
print("Padding exclusion verified: masked pad positions (even if they look like errors) do not change scores.")

# ---- 4. Error mask, first-position drop, aggregation, and zero-score cases ----
cats, first_is_err = [], 0
for i in range(2000):
    err = TOP1[i] != TARGET_IDS[i]
    first_is_err += int(err[0])
    err = err.copy(); err[0] = False                                   # ignore_bos
    if err.sum() < 2:
        cats.append("<2 error tokens → 0"); continue
    d = (TGT_LP[i] - (-BASE_LOSS_CACHED[i]))[err]
    if not (d < 0).any():
        cats.append("no negative Δ → 0 (forced)")
    elif not (d > 0).any():
        cats.append("no positive Δ → 0 (genuine)")
    else:
        v = d[d > 0].sum(dtype=np.float32) / (np.abs(d[d < 0]).sum(dtype=np.float32) + np.float32(1e-12))
        assert np.isclose(np.float32(v), EZ_DEV[i], rtol=1e-5, atol=1e-6), i
        cats.append("regular ratio")
cats = np.array(cats)
print(f"\nFirst scored position was an error in {first_is_err} texts (excluded by ignore_bos, as in the repo).")
print(pd.crosstab(pd.Series(cats, name="case"), pd.Series(np.where(LABELS == 1, "member", "non-member"), name="label")))
assert np.all(EZ_DEV[cats != "regular ratio"] == 0)

# ---- 5. Direction ----
print(f"\nMean score: members {EZ_DEV[LABELS == 1].mean():.4f} vs non-members {EZ_DEV[LABELS == 0].mean():.4f} | "
      f"AUC {roc_auc_score(LABELS, EZ_DEV):.4f} (higher = member, as in attack.py)")

Target: PeftModel = True | active adapter: default | total ||lora_B|| = 93.516
Causal shift verified: position k scores token k+1 given tokens 0..k.
Target re-run vs EZ-6 stats: max |Δlp| = 0.00e+00, top-1 agreement = 1.0000
Plain GPT-2 vs cached reference (adapter-disabled) losses: max |Δ| = 0.00e+00
Mean |target − reference| log-prob = 0.692 (adapter has an effect)
Padding exclusion verified: masked pad positions (even if they look like errors) do not change scores.

First scored position was an error in 1741 texts (excluded by ignore_bos, as in the repo).
label          member  non-member
case                             
regular ratio    1000        1000

Mean score: members 3.7598 vs non-members 2.7835 | AUC 0.6829 (higher = member, as in attack.py)
